# clean-index.ipynb

This file runs locally. It builds a clean table from the typed LABINS volume index pages in `data/labins/index-ocr/`, one row per township and range, with columns for surveyor, variation, diagram page, TWP and RGE. Only pages tagged `index` are used, not `unsure` or `diagram`.

The first step is the surveyor column. Each row can have more than one surveyor, so the clean value is a set. The first cell extracts the raw surveyor lines for each row. The second cell lists every surveyor name variant by frequency, then clusters them, to code the correct names and match the rest by similarity.

In [1]:
# imports
import re, hashlib, statistics, json
from pathlib import Path
import pandas as pd

# set dir
OCR_DIR = Path("data/labins/index-ocr")
OUT_DIR = Path("data/labins/index-clean")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# patterns
HEADER = re.compile(r"SURVEYOR\s*\(S\)", re.I)                                      # typed table header
LEAD = re.compile(r"^\s*\d{1,3}(?:\s*[,&-]\s*\d{1,3})*\s?[NSEW](?![a-z])\.?\s*")      # twp or rge token at line start
DATE = re.compile(r"\b1[6-9]\d\d(?:-\d+)?\b")
VAR = re.compile(r"\d+\s*°[^A-Za-z]*[EWew]?\b")
NOISE = re.compile(r"^(Index to|Special Indices|Range \d)|\d+(st|nd|rd|th)\b", re.I)   # title lines and page ranges
GAP = 0.3            # line gap, in median line heights, that starts a new row

# helpers
def date_x(lines, header_bottom, width):
    hits = [l for l in lines if l["text"].strip().upper().startswith("DATE") and l["box"][3] <= header_bottom + 40]
    return min(l["box"][0] for l in hits) if hits else 0.43 * width   # merged header has no separate date box

def rows_of(lines, header_bottom):
    body = sorted([l for l in lines if l["box"][1] > header_bottom - 5], key=lambda l: l["box"][1])
    if not body:
        return []
    med = statistics.median(l["box"][3] - l["box"][1] for l in body)
    rows, bottom = [], -1
    for l in body:
        if rows and l["box"][1] < bottom + GAP * med:
            rows[-1].append(l)                           # second surveyor line sits just below the first
            bottom = max(bottom, l["box"][3])
        else:
            rows.append([l])
            bottom = l["box"][3]
    return rows

def surveyor_lines(row, dx):
    out = []
    for l in sorted(row, key=lambda l: (l["box"][1], l["box"][0])):
        if l["box"][0] >= dx - 60:
            continue                                     # date, variation and page columns
        t = l["text"].strip()
        while LEAD.match(t):
            t = LEAD.sub("", t, 1)                       # twp and rge can share the line
        t = VAR.sub("", DATE.sub("", t)).strip(" ,;")
        if t and re.search(r"[A-Za-z]{2}", t) and not t.startswith("(") and not NOISE.search(t):
            out.append(t)
    return out

# extract rows
rows, n = [], {"typed": 0, "no header": 0, "duplicate": 0}
for f in sorted(OCR_DIR.glob("vol*.json")):
    seen = set()
    for p in json.loads(f.read_text(encoding="utf-8")):
        if p["tag"] != "index":
            continue
        key = hashlib.md5("\n".join(l["text"] for l in p["lines"]).encode()).hexdigest()
        if key in seen:
            n["duplicate"] += 1                          # some pdfs repeat their front pages
            continue
        seen.add(key)
        header = [l for l in p["lines"] if HEADER.search(l["text"])]
        if not header:
            n["no header"] += 1
            continue
        n["typed"] += 1
        hb = max(l["box"][3] for l in header)
        dx = date_x(p["lines"], hb, p["width"])
        for i, r in enumerate(rows_of(p["lines"], hb)):
            rows.append({"vol": f.stem, "page": p["page"], "row": i,
                         "surveyor_raw": " | ".join(surveyor_lines(r, dx)),
                         "row_text": " | ".join(l["text"] for l in sorted(r, key=lambda l: (l["box"][1], l["box"][0])))})

# save
df = pd.DataFrame(rows)
df.to_csv(OUT_DIR / "surveyor_rows.csv", index=False)
print(n)
print(len(df), "rows;", (df["surveyor_raw"] == "").sum(), "without a surveyor line;", df["surveyor_raw"].str.contains(" | ", regex=False).sum(), "with more than one")
df.head(10)

{'typed': 366, 'no header': 92, 'duplicate': 14}
3864 rows; 39 without a surveyor line; 184 with more than one


,vol,page,row,surveyor_raw,row_text
0,vol001,1,0,Benj. Clements,1S | 1824 | 1E | 3 | Benj. Clements | Not Given
1,vol001,1,1,David Thomas,2S | 1824 | 1E | 3 | Not Given | David Thomas
2,vol001,1,2,David Thomas,3S | 1824 | 4 | 1E | Not Given | David Thomas
3,vol001,1,3,David Thomas,1824 | 4S | 1E | 4 | Not Given | David Thomas
4,vol001,1,4,James M. Sloan,2S | 1847 | 1E | James M. Sloan | Not Given | 5
5,vol001,1,5,David Thomas,2S | 4E | 1848 | 5 | David Thomas | Not Given
6,vol001,1,6,LeRoy May,3S | 1E | LeRoy May | 1824 & | 6 | Not Given |...
7,vol002,1,0,David Thomas,1S | 2E | 1824 | David Thomas | Not Given | 3
8,vol002,1,1,Thomas T. Hays,Thomas T. Hays | 1824 | 1N | 3E | Not Given | ...
9,vol002,1,2,David Thomas,1S | 4E | 1824 | David Thomas | Not Given | 5


In [3]:
# imports
import re, unicodedata
import pandas as pd
from difflib import SequenceMatcher
from pathlib import Path

# set dir
OUT_DIR = Path("data/labins/index-clean")

# config
SURNAME_MIN = 0.85   # surname similarity to attach a variant to a more common one, given identical initials and given names

# load rows
rows = pd.read_csv(OUT_DIR / "surveyor_rows.csv", keep_default_na=False)

# variant counts
names = rows.assign(surveyor=rows["surveyor_raw"].str.split(" | ", regex=False)).explode("surveyor")
names = names[names["surveyor"] != ""]
freq = (names.groupby("surveyor")
        .agg(n=("vol", "size"), n_vols=("vol", "nunique"), first_vol=("vol", "min"))
        .reset_index()
        .sort_values(["n", "surveyor"], ascending=[False, True])
        .reset_index(drop=True))
print(len(freq), "variants,", freq["n"].sum(), "mentions,", (freq["n"] == 1).sum(), "seen once")

# normalize
def clean(s):
    return unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode().lower().replace("&", "and")   # drops accents

def norm(s):
    return re.sub(r"[^a-z0-9]", "", clean(s))                                # drops spacing, dots and case

def split(s):
    toks = [t for t in clean(s).split() if re.search(r"[a-z]", t)]           # drops page numbers and stray dots
    return (norm(" ".join(toks[:-1])), norm(toks[-1])) if toks else ("", "")  # initials and given names, surname

key = [norm(s) for s in freq["surveyor"]]
pre, sur = zip(*[split(s) for s in freq["surveyor"]])

# parent of each variant: most similar variant above it in the frequency order
parent, sim, how = [None] * len(freq), [None] * len(freq), [""] * len(freq)
for i in range(1, len(freq)):
    best, best_s, best_h = None, 0.0, ""
    for j in range(i):
        if key[i] == key[j]:
            s, h = 1.0, "same after normalizing"
        elif pre[i] == pre[j] and sur[i] and sur[j]:
            s, h = SequenceMatcher(None, sur[i], sur[j]).ratio(), "surname"    # initials must match exactly
            if s < SURNAME_MIN:
                continue
        else:
            continue
        if s > best_s:                                                         # ties keep the more common variant
            best, best_s, best_h = j, s, h
    if best is not None:
        parent[i], sim[i], how[i] = best, round(best_s, 3), best_h

# root and depth
root, depth = list(range(len(freq))), [0] * len(freq)
for i in range(len(freq)):
    j = i
    while parent[j] is not None:
        j = parent[j]
        depth[i] += 1
    root[i] = j

# clusters table
clusters = freq.assign(
    parent=[freq["surveyor"][p] if p is not None else "" for p in parent],
    similarity=sim,
    match=how,
    root=[freq["surveyor"][r] for r in root],
    depth=depth)
clusters["cluster_size"] = clusters.groupby("root")["surveyor"].transform("size")
clusters["cluster_n"] = clusters.groupby("root")["n"].transform("sum")
clusters = clusters.sort_values(["cluster_n", "root", "depth", "n"], ascending=[False, True, True, False]).reset_index(drop=True)

# save
freq.to_csv(OUT_DIR / "surveyor_freq.csv", index=False)
clusters.to_csv(OUT_DIR / "surveyor_clusters.csv", index=False)
print(clusters["root"].nunique(), "clusters,", clusters.loc[clusters["cluster_size"] > 1, "root"].nunique(), "with more than one variant")
clusters[clusters["cluster_size"] > 1].head(60)

274 variants, 4017 mentions, 94 seen once
219 clusters, 45 with more than one variant


,surveyor,n,n_vols,first_vol,parent,similarity,match,root,depth,cluster_size,cluster_n
0,Henry Washington,293,14,vol003,,NaN,,Henry Washington,0,2,295
1,Henry Washington (3),2,1,vol078,Henry Washington,1.000,surname,Henry Washington,1,2,295
2,A.M. Randolph,198,18,vol058,,NaN,,A.M. Randolph,0,4,216
3,A. M. Randolph,16,1,vol016,A.M. Randolph,1.000,same after normalizing,A.M. Randolph,1,4,216
4,A.M. Randolph ..,1,1,vol190,A.M. Randolph,1.000,same after normalizing,A.M. Randolph,1,4,216
5,A.M.Randolph,1,1,vol181,A.M. Randolph,1.000,same after normalizing,A.M. Randolph,1,4,216
6,J. W. Exum,111,8,vol012,,NaN,,J. W. Exum,0,3,177
7,J.W. Exum,64,8,vol009,J. W. Exum,1.000,same after normalizing,J. W. Exum,1,3,177
8,J.W.Exum,2,2,vol061,J. W. Exum,1.000,same after normalizing,J. W. Exum,1,3,177
10,M.A. Williams,119,15,vol061,,NaN,,M.A. Williams,0,2,121


In [6]:
# imports
import pandas as pd
from pathlib import Path

# set dir
OUT_DIR = Path("data/labins/index-clean")

# load
clusters = pd.read_csv(OUT_DIR / "surveyor_clusters.csv", keep_default_na=False)

# one row per cluster: parent, then its children
rows = []
for root, g in clusters.groupby("root"):
    kids = g[g["surveyor"] != root].sort_values("n", ascending=False)
    rows.append({
        "parent": root,
        "parent_n": int(g.loc[g["surveyor"] == root, "n"].iloc[0]),
        "n_children": len(kids),
        "children": "; ".join(f"{s} ({n})" for s, n in zip(kids["surveyor"], kids["n"])),
        "cluster_n": int(g["n"].sum()),
    })
table = pd.DataFrame(rows).sort_values("cluster_n", ascending=False).reset_index(drop=True)

# show all
pd.set_option("display.max_rows", None, "display.max_colwidth", None, "display.width", 250)
print(len(table), "clusters")
table

219 clusters


,parent,parent_n,n_children,children,cluster_n
0,Henry Washington,293,1,Henry Washington (3) (2),295
1,A.M. Randolph,198,3,A. M. Randolph (16); A.M. Randolph .. (1); A.M.Randolph (1),216
2,J. W. Exum,111,2,J.W. Exum (64); J.W.Exum (2),177
3,A.H. Jones,155,0,,155
4,M.A. Williams,119,1,M.A. Williams .. (2),121
5,B & J. B. Clements,61,2,B & J.B. Clements (43); B. & J. B. Clements (14),118
6,John Jackson,103,1,JohnJackson (1),104
7,Benj. Clements,99,0,,99
8,Sam Reid,94,0,,94
9,C.C. Tracy,84,1,C.C.Tracy (1),85
